# ARC-AGI-3 — V4 + action_proposer (xinxiang000)

Architecture: scipy perception + SmolLM3-3B `/no_think` + Reflection schema validation + action_proposer K=3.

5-game G_base local result: **mean change_rate 82%, 0/5 wins** (see `presentation/report.md`).

In [ ]:
# Cell 1: install arc-agi + dotenv from the competition's offline wheel cache
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
# v4 + action_proposer agent wrapping arc_agent.ActionAgent for the Kaggle Agent harness
import os, sys
sys.path.insert(0, '/kaggle/input/arcagi3-code')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

import random
from typing import Any
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

_BACKBONE = None
_LOAD_ERROR = None

def _load_backbone():
    global _BACKBONE, _LOAD_ERROR
    if _BACKBONE is not None or _LOAD_ERROR is not None:
        return _BACKBONE
    try:
        from arc_agent.vlm_backbone import make_backbone
        _BACKBONE = make_backbone('/kaggle/input/smollm3-3b-4bit',
                                   reasoning_mode='no_think')
    except Exception as e:
        _LOAD_ERROR = e
        print(f'[my_agent] backbone load FAILED: {e}', flush=True)
    return _BACKBONE


class MyAgent(Agent):
    """V4 + action_proposer wrapped for the Kaggle Agent harness."""
    MAX_ACTIONS = float('inf')

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        from arc_agent.agents.action_agent import ActionAgent
        bb = _load_backbone()
        if bb is not None:
            self._aa = ActionAgent(backbone=bb, seed=42, max_new_tokens=256)
            self._aa.use_proposer = True
        else:
            self._aa = None
        random.seed(hash(self.game_id) % (1 << 31))

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _random_action(self) -> GameAction:
        a = random.choice([x for x in GameAction if x is not GameAction.RESET])
        if a.is_complex():
            a.set_data({'x': random.randint(0, 63), 'y': random.randint(0, 63)})
        return a

    def choose_action(self, frames: list[FrameData],
                       latest_frame: FrameData) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            a = GameAction.RESET
            a.reasoning = 'reset (game over or not played)'
            return a

        if self._aa is None:
            a = self._random_action()
            a.reasoning = 'fallback random (backbone load failed)'
            return a

        try:
            action, reasoning = self._aa.choose(latest_frame, frames)
        except Exception as e:
            action = self._random_action()
            reasoning = f'fallback random (agent error: {type(e).__name__})'

        if action.is_simple():
            action.reasoning = reasoning or 'no reasoning'
        elif action.is_complex():
            action.reasoning = {
                'desired_action': action.value,
                'my_reason': reasoning or 'no reasoning',
            }
        return action


In [ ]:
# Cell 3: scoring run -- only when Kaggle is reranking the notebook with the private test gateway.
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway to be ready (per Kaggle starter pattern)
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the competition repo to a writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent file in
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Rewrite agents/__init__.py so only the two agents we need import.
    # (Stock __init__ eagerly imports langchain / smolagents templates that
    # are not installed in this environment.)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
''')

    # Write .env that points the harness at gateway:8001
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # Run
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

In [ ]:
# Cell 4: dummy submission.parquet for the commit-only path (non-rerun).
# Real scoring is server-side via the gateway during competition rerun.
import pandas as pd
import os

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Wrote dummy submission.parquet (real scoring happens via gateway in rerun)')